# Ablation Study : RAG baseline vs LightRAG hybride vs Agentic GraphRAG

**PFE Agentic GraphRAG — Wiame Anejjar — Master SDIA 2025-2026**

## Objectif

Isoler la contribution de chaque couche de l'architecture en évaluant **trois systèmes** sur le **même benchmark**, avec le **même LLM générateur** et les **mêmes métriques RAGAS** :

1. **RAG vectoriel (ChromaDB seul)** — aucun graphe, retrieval par similarité vectorielle uniquement.
2. **LightRAG hybride (sans agent)** — retrieval graphe + vecteurs de LightRAG (mode hybrid), une seule génération, sans boucle de correction.
3. **Agentic GraphRAG (contribution du PFE)** — même retrieval LightRAG que (2), plus la boucle CRITIQUE → SELF_CORRECT avec juge indépendant.

La seule variable qui change entre (2) et (3) est la présence de la couche agentique : cela permet de mesurer sa contribution propre, isolée du choix du backend de retrieval.

**Benchmark utilisé** : `data/processed/benchmark_true_multihop.json` — 76 questions vrai multi-hop, validées par analyse de contenu (voir `scripts/validate_multihop_benchmark.py`), pas par la simple étiquette `hop_type` d'origine.

**Important** : les résultats de ce notebook sont ceux effectivement obtenus à l'exécution , aucun chiffre n'est pré-rempli ou supposé à l'avance.

In [ ]:
import os, sys, json, random, asyncio, time, datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
pd.set_option("display.max_colwidth", 120)
print(f"Notebook exécuté le : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Chargement du benchmark

In [ ]:
BENCHMARK_PATH = Path("data/processed/benchmark_true_multihop.json")
N_QUESTIONS = 20
SEED = 42

with BENCHMARK_PATH.open("r", encoding="utf-8") as f:
    all_items = json.load(f)

random.seed(SEED)
eval_items = random.sample(all_items, min(N_QUESTIONS, len(all_items)))

print(f"Benchmark source                     : {BENCHMARK_PATH}")
print(f"Questions vrai multi-hop disponibles  : {len(all_items)}")
print(f"Questions échantillonnées (seed={SEED})    : {len(eval_items)}")

display(pd.DataFrame([
    {"question": it["question"], "ground_truth": it["ground_truth"]}
    for it in eval_items
]))

## 2. Configuration commune (générateur + prompt)

Pour que la comparaison isole bien la variable étudiée (retrieval + boucle agentique), les trois systèmes réutilisent **directement** le générateur et le prompt de `src/agent/graph_v3.py` (fonction `_llm_call`, prompt `RESPONSE_SYSTEM_PROMPT`) — pas une copie séparée. Quel que soit le backend actif (NVIDIA / Groq / Ollama local, contrôlé par les variables d'environnement `USE_NVIDIA`/`USE_GROQ`), les trois systèmes utilisent donc exactement le même LLM pour la génération finale.

Seuls le contexte fourni (retrieval) et la présence ou non de la boucle CRITIQUE/SELF_CORRECT diffèrent entre les trois systèmes.

In [ ]:
from langchain_ollama import OllamaEmbeddings

# Réutilise l'infrastructure déjà initialisée dans graph_v3.py (déclenche l'init
# du LLM générateur, du juge et de rag_instance -> un seul point de configuration)
from src.agent import graph_v3

EMBED_MODEL    = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL     = os.getenv("OLLAMA_URL", "http://localhost:11434")

if graph_v3.USE_NVIDIA and graph_v3.NVIDIA_API_KEY:
    GENERATOR_DESC = f"NVIDIA {graph_v3.NVIDIA_MODEL}"
elif graph_v3.USE_GROQ and graph_v3.GROQ_API_KEY:
    GENERATOR_DESC = f"Groq {graph_v3.GROQ_GENERATOR_MODEL}"
else:
    GENERATOR_DESC = f"Ollama {graph_v3.MODEL_NAME} (num_ctx={graph_v3.OLLAMA_NUM_CTX})"

def generate_answer(question: str, context: str) -> str:
    # Meme troncature defensive que node_response (evite les 413 "request too
    # large" de certains backends, ex. Groq llama-3.1-8b-instant)
    context = context[:graph_v3.MAX_GENERATOR_CONTEXT_CHARS]
    user = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer (use ONLY the context above, cite sources):"
    return graph_v3._llm_call(graph_v3.RESPONSE_SYSTEM_PROMPT, user)

print(f"Générateur commun (systèmes 1, 2 et 3) : {GENERATOR_DESC}")
print(f"Contexte tronqué à {graph_v3.MAX_GENERATOR_CONTEXT_CHARS} caractères avant génération (anti-413)")

## 3. Système 1 — RAG vectoriel (ChromaDB seul)

In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR        = os.getenv("CHROMA_DIR", "indexes/chroma_pfe500_baseline")
CHROMA_COLLECTION = "pfe_500_baseline"
TOP_K_VECTOR      = int(os.getenv("TOP_K_VECTOR", "5"))

_baseline_emb = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL)
_vectorstore  = Chroma(persist_directory=CHROMA_DIR, embedding_function=_baseline_emb,
                       collection_name=CHROMA_COLLECTION)
_baseline_retriever = _vectorstore.as_retriever(search_kwargs={"k": TOP_K_VECTOR})

print(f"ChromaDB         : {CHROMA_DIR} | collection={CHROMA_COLLECTION}")
print(f"Vecteurs stockés : {_vectorstore._collection.count()}")
print(f"top_k            : {TOP_K_VECTOR}")

def run_rag_baseline(question: str) -> dict:
    t0 = time.time()
    docs = _baseline_retriever.invoke(question)
    contexts = [d.page_content for d in docs]
    context_str = "\n\n".join(f"[Doc {i+1}] {c[:1200]}" for i, c in enumerate(contexts))
    answer = generate_answer(question, context_str)
    return {"answer": answer,
            "contexts": contexts if contexts else ["No context retrieved."],
            "context_chars": len(context_str),
            "latency_s": round(time.time() - t0, 2)}

In [ ]:
results_baseline = []
ragas_data_baseline = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("=== SYSTÈME 1 : RAG BASELINE (ChromaDB) ===")
for i, ex in enumerate(eval_items, 1):
    print(f"[{i}/{len(eval_items)}] {ex['question'][:70]}")
    r = run_rag_baseline(ex["question"])
    ragas_data_baseline["question"].append(ex["question"])
    ragas_data_baseline["answer"].append(r["answer"])
    ragas_data_baseline["contexts"].append(r["contexts"])
    ragas_data_baseline["ground_truth"].append(ex["ground_truth"])
    results_baseline.append({"question": ex["question"], "answer": r["answer"][:200],
                              "context_chars": r["context_chars"], "latency_s": r["latency_s"]})
    print(f"  -> {r['answer'][:100]}  ({r['latency_s']}s)")

df_details_baseline = pd.DataFrame(results_baseline)

## 4. Système 2 — LightRAG hybride (retrieval seul, sans agent)

Réutilise directement `rag_instance` de `src/agent/graph_v3.py` (même index post-traité, mêmes paramètres `top_k`/`chunk_top_k`/mode) — garantit un retrieval strictement identique à celui du système 3. Seule différence : un unique appel de génération, sans CRITIQUE ni SELF_CORRECT.

In [ ]:
from lightrag import QueryParam

print(f"[LightRAG] index réutilisé : {graph_v3.INDEX_DIR}")
print(f"[LightRAG] top_k={graph_v3.TOP_K_LIGHTRAG} | chunk_top_k={graph_v3.CHUNK_TOP_K} | mode=hybrid")

def run_lightrag_only(question: str) -> dict:
    t0 = time.time()

    async def _query():
        return await graph_v3.rag_instance.aquery_data(
            question,
            param=QueryParam(
                mode="hybrid",
                top_k=graph_v3.TOP_K_LIGHTRAG,
                chunk_top_k=graph_v3.CHUNK_TOP_K,
                enable_rerank=False,
                max_total_tokens=int(os.getenv("MAX_TOTAL_TOKENS", "12000")),
            ),
        )

    result = asyncio.get_event_loop().run_until_complete(_query())
    data = (result or {}).get("data", {}) if isinstance(result, dict) else {}
    context_str = graph_v3._format_context_from_raw(data)
    contexts    = graph_v3._contexts_list_from_raw(data)
    answer = generate_answer(question, context_str)  # UN SEUL appel, pas de critique/self-correct
    return {"answer": answer, "contexts": contexts,
            "context_chars": len(context_str),
            "latency_s": round(time.time() - t0, 2)}

In [ ]:
results_lightrag = []
ragas_data_lightrag = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("=== SYSTÈME 2 : LIGHTRAG HYBRIDE (SANS AGENT) ===")
for i, ex in enumerate(eval_items, 1):
    print(f"[{i}/{len(eval_items)}] {ex['question'][:70]}")
    r = run_lightrag_only(ex["question"])
    ragas_data_lightrag["question"].append(ex["question"])
    ragas_data_lightrag["answer"].append(r["answer"])
    ragas_data_lightrag["contexts"].append(r["contexts"])
    ragas_data_lightrag["ground_truth"].append(ex["ground_truth"])
    results_lightrag.append({"question": ex["question"], "answer": r["answer"][:200],
                              "context_chars": r["context_chars"], "latency_s": r["latency_s"]})
    print(f"  -> {r['answer'][:100]}  ({r['latency_s']}s)")

df_details_lightrag = pd.DataFrame(results_lightrag)
print(f"\nTaille moyenne du contexte fusionné : {df_details_lightrag['context_chars'].mean():.0f} caractères "
      f"(max={df_details_lightrag['context_chars'].max()})")

## 5. Système 3 — Agentic GraphRAG (contribution du PFE)

Boucle complète : QUERY → HYBRID_SEARCH (LightRAG, identique au système 2) → RESPONSE → CRITIQUE (juge indépendant) → FINALIZE | SELF_CORRECT (max 3 itérations). Voir `src/agent/graph_v3.py::run_agent`.

In [ ]:
results_agentic = []
ragas_data_agentic = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("=== SYSTÈME 3 : AGENTIC GRAPHRAG ===")
for i, ex in enumerate(eval_items, 1):
    print(f"[{i}/{len(eval_items)}] {ex['question'][:70]}")
    t0 = time.time()
    result = graph_v3.run_agent(ex["question"])
    latency = round(time.time() - t0, 2)

    ragas_data_agentic["question"].append(ex["question"])
    ragas_data_agentic["answer"].append(result.get("final_response", ""))
    ragas_data_agentic["contexts"].append(result.get("lightrag_retrieved_contexts") or ["No context retrieved."])
    ragas_data_agentic["ground_truth"].append(ex["ground_truth"])

    results_agentic.append({
        "question": ex["question"],
        "answer": result.get("final_response", "")[:200],
        "context_chars": len(result.get("lightrag_context", "")),
        "critique_score": result.get("critique_score", 0.0),
        "judge_independent": result.get("critique_judge_independent", False),
        "iterations": result.get("iteration", 0),
        "latency_s": latency,
    })
    print(f"  -> score={result.get('critique_score', 0):.2f} | iter={result.get('iteration', 0)} ({latency}s)")

df_details_agentic = pd.DataFrame(results_agentic)
n_self_eval = (~df_details_agentic["judge_independent"]).sum()
if n_self_eval:
    print(f"\n⚠ {n_self_eval}/{len(df_details_agentic)} scores de critique sont des auto-évaluations (judge_independent=False)")

## 6. Évaluation RAGAS des 3 systèmes

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama

if graph_v3.USE_GROQ and graph_v3.GROQ_API_KEY:
    from langchain_groq import ChatGroq
    _ragas_groq_model = os.getenv("RAGAS_GROQ_MODEL", "llama-3.1-8b-instant")
    print(f"[RAGAS JUDGE] Groq {_ragas_groq_model}")
    _ragas_chat_model = ChatGroq(model=_ragas_groq_model, api_key=graph_v3.GROQ_API_KEY, temperature=0)
    _ragas_batch_size = 4  # Groq supporte la concurrence, contrairement a Ollama local
elif graph_v3.USE_NVIDIA and graph_v3.NVIDIA_API_KEY:
    from langchain_openai import ChatOpenAI
    print(f"[RAGAS JUDGE] NVIDIA {graph_v3.NVIDIA_MODEL}")
    _ragas_chat_model = ChatOpenAI(model=graph_v3.NVIDIA_MODEL, api_key=graph_v3.NVIDIA_API_KEY,
                                    base_url=graph_v3.NVIDIA_BASE_URL, temperature=0)
    _ragas_batch_size = 1
else:
    print(f"[RAGAS JUDGE] Ollama {graph_v3.MODEL_NAME} (local)")
    _ragas_chat_model = ChatOllama(model=graph_v3.MODEL_NAME, base_url=OLLAMA_URL, temperature=0)
    _ragas_batch_size = 1

_ragas_llm = LangchainLLMWrapper(_ragas_chat_model)
_ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL))

def run_ragas(ragas_data: dict, label: str) -> pd.DataFrame:
    print(f"\nRAGAS -> {label} ({len(ragas_data['question'])} questions, batch_size={_ragas_batch_size})...")
    dataset = Dataset.from_dict(ragas_data)
    result = evaluate(
        dataset,
        metrics=[
            Faithfulness(llm=_ragas_llm),
            AnswerRelevancy(llm=_ragas_llm, embeddings=_ragas_emb),
            ContextPrecision(llm=_ragas_llm),
            ContextRecall(llm=_ragas_llm),
        ],
        run_config=RunConfig(timeout=180, max_retries=3, max_wait=30),
        batch_size=_ragas_batch_size,
        raise_exceptions=False,
    )
    df = result.to_pandas()
    df["system"] = label
    return df

df_ragas_baseline = run_ragas(ragas_data_baseline, "RAG baseline (ChromaDB)")
df_ragas_lightrag = run_ragas(ragas_data_lightrag, "LightRAG hybride (sans agent)")
df_ragas_agentic  = run_ragas(ragas_data_agentic,  "Agentic GraphRAG")

## 7. Tableau comparatif des résultats

In [ ]:
METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
SYSTEM_ORDER = ["RAG baseline (ChromaDB)", "LightRAG hybride (sans agent)", "Agentic GraphRAG"]

all_ragas = pd.concat([df_ragas_baseline, df_ragas_lightrag, df_ragas_agentic], ignore_index=True)

comparison_table = all_ragas.groupby("system")[METRICS].mean().reindex(SYSTEM_ORDER).round(3)
comparison_table["latency_moy_s"] = [
    df_details_baseline["latency_s"].mean(),
    df_details_lightrag["latency_s"].mean(),
    df_details_agentic["latency_s"].mean(),
]

print(f"=== Tableau comparatif — moyennes RAGAS sur {len(eval_items)} questions ===")
display(comparison_table)

Path("Eval_agentic").mkdir(parents=True, exist_ok=True)
comparison_table.to_csv("Eval_agentic/ablation_study_comparaison.csv")
all_ragas.to_csv("Eval_agentic/ablation_study_details_par_question.csv", index=False)
print("\n✓ Eval_agentic/ablation_study_comparaison.csv")
print("✓ Eval_agentic/ablation_study_details_par_question.csv")

In [ ]:
ax = comparison_table[METRICS].plot(kind="bar", figsize=(11, 6), rot=15)
ax.set_ylabel("Score RAGAS (0-1)")
ax.set_title("Ablation study — RAG baseline vs LightRAG hybride vs Agentic GraphRAG")
ax.set_ylim(0, 1)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()

Path("figures").mkdir(parents=True, exist_ok=True)
plt.savefig("figures/fig15_ablation_study_comparaison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ figures/fig15_ablation_study_comparaison.png")

## 8. Paramètres utilisés par système

In [ ]:
judge_desc = (
    f"Groq {graph_v3.GROQ_JUDGE_MODEL} (indépendant)"
    if graph_v3.USE_GROQ_JUDGE and graph_v3.GROQ_API_KEY
    else f"Ollama {graph_v3.JUDGE_MODEL_NAME} (indépendant)"
)

params_table = pd.DataFrame([
    {
        "Système": "RAG baseline (ChromaDB)",
        "Backend retrieval": f"ChromaDB — {CHROMA_DIR}",
        "Corpus indexé": f"{_vectorstore._collection.count()} chunks (nomic-embed-text, chunk_size=900, overlap=120, 500 documents)",
        "Paramètres retrieval": f"top_k={TOP_K_VECTOR}",
        "Générateur": GENERATOR_DESC,
        "Boucle agentique": "Non",
    },
    {
        "Système": "LightRAG hybride (sans agent)",
        "Backend retrieval": f"LightRAG — {graph_v3.INDEX_DIR} (post-traité : fusion entités + arêtes de co-occurrence)",
        "Corpus indexé": "4945 nœuds / 5593 relations (après post-traitement du 22/07/2026)",
        "Paramètres retrieval": f"mode=hybrid, top_k={graph_v3.TOP_K_LIGHTRAG}, chunk_top_k={graph_v3.CHUNK_TOP_K}, enable_rerank=False",
        "Générateur": GENERATOR_DESC,
        "Boucle agentique": "Non (1 seul appel de génération)",
    },
    {
        "Système": "Agentic GraphRAG",
        "Backend retrieval": f"LightRAG — {graph_v3.INDEX_DIR} (identique au système 2)",
        "Corpus indexé": "4945 nœuds / 5593 relations (identique au système 2)",
        "Paramètres retrieval": f"mode=hybrid, top_k={graph_v3.TOP_K_LIGHTRAG}, chunk_top_k={graph_v3.CHUNK_TOP_K}, enable_rerank=False",
        "Générateur": GENERATOR_DESC,
        "Boucle agentique": f"Oui — juge={judge_desc}, seuil_critique={graph_v3.CRITIQUE_SEUIL}, max_iterations={graph_v3.MAX_ITERATIONS}",
    },
])

display(params_table)
params_table.to_csv("Eval_agentic/ablation_study_parametres.csv", index=False)
print("✓ Eval_agentic/ablation_study_parametres.csv")

## 9. Limites et discussion

In [ ]:
best_system = comparison_table[METRICS].mean(axis=1).idxmax()
delta_vs_baseline = (comparison_table.loc["Agentic GraphRAG", METRICS] - comparison_table.loc["RAG baseline (ChromaDB)", METRICS]).round(3)
delta_vs_lightrag = (comparison_table.loc["Agentic GraphRAG", METRICS] - comparison_table.loc["LightRAG hybride (sans agent)", METRICS]).round(3)

print("=== Lecture automatique des résultats obtenus (calculée, pas supposée) ===\n")
print(f"Générateur utilisé pour les 3 systèmes : {GENERATOR_DESC}\n")
print(f"Système avec la moyenne RAGAS la plus élevée : {best_system}\n")
print("Delta Agentic GraphRAG - RAG baseline :")
print(delta_vs_baseline.to_string())
print("\nDelta Agentic GraphRAG - LightRAG hybride (sans agent) :")
print(delta_vs_lightrag.to_string())
print(f"\nTaille moyenne du contexte (système 2/3, LightRAG) : {df_details_lightrag['context_chars'].mean():.0f} caractères (max={df_details_lightrag['context_chars'].max()})")
print(f"Taille moyenne du contexte (système 1, ChromaDB)   : {df_details_baseline['context_chars'].mean():.0f} caractères (max={df_details_baseline['context_chars'].max()})")
if not (graph_v3.USE_NVIDIA and graph_v3.NVIDIA_API_KEY) and not (graph_v3.USE_GROQ and graph_v3.GROQ_API_KEY):
    print(f"Fenêtre de contexte du générateur local (num_ctx) : {graph_v3.OLLAMA_NUM_CTX} tokens — "
          f"à comparer à la taille de contexte mesurée ci-dessus")
print("""
Interprétation à adapter selon le résultat réel ci-dessus :
- Si les deltas sont POSITIFS sur les 4 métriques : la boucle agentique (critique
  + self-correct) apporte un gain mesurable au-delà du graphe seul -> résultat
  à mettre en avant tel quel dans le mémoire.
- Si LightRAG seul égale ou dépasse l'Agentic sur certaines métriques : cela
  peut s'expliquer par la taille limitée du corpus (500 documents) qui laisse
  peu de marge au SELF_CORRECT pour retrouver un meilleur contexte -> à
  documenter comme limite du corpus, pas de l'architecture.
- Si les 3 systèmes sont proches : le goulot d'étranglement est probablement
  le retrieval / la qualité du graphe plutôt que la couche agentique -> cf.
  discussion ci-dessous.
""")

### Limites identifiées

**Indexation et qualité du graphe**
- Le graphe LightRAG a été post-traité (fusion de 103 clusters de doublons, ajout de 498 arêtes de co-occurrence) mais reste **peu dense** : densité 0.00046, degré moyen 2.26, composante géante à 55.3 % des nœuds seulement (contre 43.3 % avant post-traitement). Une partie des questions peut donc rester sans chemin de graphe direct entre les entités concernées, même après amélioration.
- Le corpus indexé (500 résumés arXiv en cs.AI) est **volontairement restreint** bien adapté à un PFE mais trop petit pour que l'avantage du multi-hop agentique se manifeste pleinement (peu de chaînes de raisonnement alternatives disponibles en cas de premier échec).

**Benchmark et évaluation**
- Échantillon de 20 questions tirées des 76 questions vrai multi-hop validées par contenu (sur 340 candidates initiales)  suffisant pour dégager une tendance.


**Modèle et infrastructure**
- Le générateur effectivement utilisé (affiché en section 2 et à la cellule précédente) a sa propre limite de fenêtre de contexte ; si le contexte fusionné mesuré ci-dessus s'en approche ou la dépasse, cela peut dégrader Faithfulness indépendamment de la qualité du retrieval lui-même.
- Le juge RAGAS (métriques Faithfulness/Context Precision/Context Recall) utilise le même modèle pour les trois systèmes  cohérent pour la comparaison relative entre systèmes, mais signifie que les scores absolus dépendent aussi des capacités de jugement propres à ce modèle.

**Ce que cette étude démontre malgré ces limites**
Le protocole isole correctement la variable étudiée : système 2 et système 3 partagent strictement le même retrieval (même index, mêmes paramètres, mêmes appels) et le même générateur , toute différence entre les deux résultats est donc attribuable à la boucle agentique (critique + self-correction), et non à un effet de retrieval ou de modèle confondu.